In [ ]:
# 6) Identifique e trate inconsistências na base (campos nulos, valores fora do padrão). Documente o que foi encontrado e como você tratou cada caso.

import pandas as pd
import numpy as np

tabelas = {
    'clientes': pd.read_csv('../data/clientes.csv'),
    'produtos': pd.read_csv('../data/produtos.csv'),
    'pedidos': pd.read_csv('../data/pedidos.csv'),
    'itens_pedido': pd.read_csv('../data/itens_pedido.csv'),
    'avaliacoes': pd.read_csv('../data/avaliacoes.csv'),
    'tickets_suporte': pd.read_csv('../data/tickets_suporte.csv')
}

print("="*80)
print("🚀 INICIANDO VARREDURA E LIMPEZA DINÂMICA DE DADOS (DATA QUALITY)")
print("="*80)

tabelas_limpas = {}

for nome, df in tabelas.items():
    print("\n" + "="*80)
    print(f"🔎 ANALISANDO TABELA: {nome.upper()}")
    print("="*80)
    
    df_clean = df.copy()
    teve_erro = False
    
    duplicatas = df_clean.duplicated().sum()
    if duplicatas > 0:
        teve_erro = True
        print(f"\n[🚨 DUPLICATAS] Encontradas {duplicatas} linhas exatamente iguais.")
        print("Antes (Exemplo de Duplicata):")
        display(df_clean[df_clean.duplicated(keep=False)].head(2))
        df_clean = df_clean.drop_duplicates()
        print("-> AÇÃO: Linhas duplicadas removidas com sucesso.")
        
    nulos_por_coluna = df_clean.isnull().sum()
    colunas_com_nulos = nulos_por_coluna[nulos_por_coluna > 0]
    
    if len(colunas_com_nulos) > 0:
        teve_erro = True
        print(f"\n[🚨 VALORES NULOS] Encontrados nulos nas seguintes colunas: {colunas_com_nulos.to_dict()}")
        
        for col in colunas_com_nulos.index:
            amostra_nulos = df_clean[df_clean[col].isnull()].head(2)
            
            if col == 'valor_total' and nome == 'pedidos':
                print(f"\nAntes (Nulos em {col}):")
                display(amostra_nulos)
                df_clean = df_clean.dropna(subset=[col])
                print(f"-> AÇÃO: Linhas sem '{col}' foram CORTADAS (impossível inferir receita).")
                
            elif col == 'desconto_aplicado' and nome == 'itens_pedido':
                print(f"\nAntes (Nulos em {col}):")
                display(amostra_nulos)
                df_clean[col] = df_clean[col].fillna(0.0)
                print("-> AÇÃO: Nulos substituídos por 0.0 (Assumiu-se que não houve desconto).")
                
            elif col == 'comentario' and nome == 'avaliacoes':
                df_clean[col] = df_clean[col].fillna('Sem comentário')
                print(f"-> AÇÃO: Nulos em '{col}' preenchidos com texto 'Sem comentário'.")
                
            elif col == 'data_resolucao' and nome == 'tickets_suporte':
                print(f"-> AÇÃO: Nulos em '{col}' mantidos. Representam tickets ainda ABERTOS.")
            else:
                df_clean = df_clean.dropna(subset=[col])
                print(f"-> AÇÃO: Linhas com nulos em '{col}' foram removidas por segurança.")

    colunas_numericas = df_clean.select_dtypes(include=[np.number]).columns
    for col in colunas_numericas:
        if not str(col).endswith('id') and col != 'id':
            negativos = df_clean[df_clean[col] < 0]
            if len(negativos) > 0:
                teve_erro = True
                print(f"\n[🚨 INCONSISTÊNCIA NUMÉRICA] {len(negativos)} valores NEGATIVOS na coluna '{col}'.")
                print("Antes:")
                display(negativos.head(2))
                df_clean[col] = df_clean[col].abs()
                print("-> AÇÃO: Convertido para valor absoluto (positivo), corrigindo possível erro de digitação.")
                print("Depois:")
                display(df_clean.loc[negativos.index].head(2))

    colunas_texto = df_clean.select_dtypes(include=['object', 'string']).columns
    
    for col in colunas_texto:
        df_clean[col] = df_clean[col].astype(str).str.strip()
        
        if 'data' in col:
            erros_data_antes = df_clean[col].isnull().sum()
            df_clean[col] = pd.to_datetime(df_clean[col], errors='coerce')
            erros_data_depois = df_clean[col].isnull().sum()
            if erros_data_depois > erros_data_antes:
                teve_erro = True
                print(f"\n[🚨 DATA INVÁLIDA] Formatos de data impossíveis encontrados na coluna '{col}'.")
                df_clean = df_clean.dropna(subset=[col])
                print("-> AÇÃO: Linhas com datas corrompidas foram CORTADAS.")

        if col == 'status':
            status_estranhos = df_clean[~df_clean['status'].str.lower().isin(['entregue', 'cancelado', 'em_transito', 'aguardando_pagamento', 'devolvido', 'aberto', 'fechado', 'em_andamento'])]
            if len(status_estranhos) > 0:
                teve_erro = True
                print(f"\n[🚨 STATUS FORA DO PADRÃO] Encontrados nomes não-oficiais na coluna '{col}':")
                print(status_estranhos['status'].value_counts().to_dict())
                df_clean['status'] = df_clean['status'].str.lower().replace({'resolvido': 'fechado', 'escalado': 'em_andamento'})
                print("-> AÇÃO: Status padronizados para a nomenclatura oficial.")

    if not teve_erro:
        print(f"✅ Nenhuma anomalia crítica encontrada. A tabela '{nome}' está íntegra.")
        
    tabelas_limpas[nome] = df_clean

print("\n" + "="*80)
print("💾 EXPORTANDO ARQUIVOS LIMPOS PARA USO NAS QUESTÕES 1 A 5...")
print("="*80)

tabelas_limpas['clientes'].to_csv('../data/clientes_limpo.csv', index=False)
tabelas_limpas['produtos'].to_csv('../data/produtos_limpo.csv', index=False)
tabelas_limpas['pedidos'].to_csv('../data/pedidos_limpo.csv', index=False)
tabelas_limpas['itens_pedido'].to_csv('../data/itens_pedido_limpo.csv', index=False)
tabelas_limpas['avaliacoes'].to_csv('../data/avaliacoes_limpo.csv', index=False)
tabelas_limpas['tickets_suporte'].to_csv('../data/tickets_suporte_limpo.csv', index=False)

print("🎉 SUCESSO! Arquivos '_limpo.csv' gerados na pasta '../data/'.")

🚀 INICIANDO VARREDURA E LIMPEZA DINÂMICA DE DADOS (DATA QUALITY)

🔎 ANALISANDO TABELA: CLIENTES
✅ Nenhuma anomalia crítica encontrada. A tabela 'clientes' está íntegra.

🔎 ANALISANDO TABELA: PRODUTOS
✅ Nenhuma anomalia crítica encontrada. A tabela 'produtos' está íntegra.

🔎 ANALISANDO TABELA: PEDIDOS

[🚨 VALORES NULOS] Encontrados nulos nas seguintes colunas: {'valor_total': 79}

Antes (Nulos em valor_total):


,id,cliente_id,data_pedido,status,valor_total,canal_venda,cupom_desconto
106,107,2567,2023-01-24,entregue,NaN,app,sim
203,204,1660,2024-12-22,entregue,NaN,site,não


-> AÇÃO: Linhas sem 'valor_total' foram CORTADAS (impossível inferir receita).

🔎 ANALISANDO TABELA: ITENS_PEDIDO

[🚨 VALORES NULOS] Encontrados nulos nas seguintes colunas: {'desconto_aplicado': 301}

Antes (Nulos em desconto_aplicado):


,id,pedido_id,produto_id,quantidade,preco_praticado,desconto_aplicado
99,100,41,40,4,454.44,NaN
107,108,45,47,4,1485.05,NaN


-> AÇÃO: Nulos substituídos por 0.0 (Assumiu-se que não houve desconto).

🔎 ANALISANDO TABELA: AVALIACOES

[🚨 VALORES NULOS] Encontrados nulos nas seguintes colunas: {'comentario': 1679}
-> AÇÃO: Nulos em 'comentario' preenchidos com texto 'Sem comentário'.

🔎 ANALISANDO TABELA: TICKETS_SUPORTE

[🚨 VALORES NULOS] Encontrados nulos nas seguintes colunas: {'data_resolucao': 1653}
-> AÇÃO: Nulos em 'data_resolucao' mantidos. Representam tickets ainda ABERTOS.

[🚨 STATUS FORA DO PADRÃO] Encontrados nomes não-oficiais na coluna 'status':
{'resolvido': 2347, 'escalado': 847}
-> AÇÃO: Status padronizados para a nomenclatura oficial.

💾 EXPORTANDO ARQUIVOS LIMPOS PARA USO NAS QUESTÕES 1 A 5...
🎉 SUCESSO! Arquivos '_limpo.csv' gerados na pasta '../data/'.
